In [7]:
import numpy as np
import pandas as pd
import pulp
import matplotlib.pyplot as plt
import highspy


In [2]:
components = ["C1", "C2", "C3"]
grades = ["Super", "Premium", "Extra"]
cost = [12, 10, 14]                 
availability = [4500, 2700, 3500]   
price = [23, 20, 18]                
min_production = 3000               
min_c1 = [0.5, 0.4, 0.6]            
max_c2_super = 0.3
max_c3_premium = 0.25
min_c2_extra = 0.1

In [3]:
def solve_petroleum(components, grades, cost, availability, price, min_production, min_c1, max_c2_super, max_c3_premium, min_c2_extra):
    cost = np.asarray(cost, dtype=float)
    availability = np.asarray(availability, dtype=float)
    price = np.asarray(price, dtype=float)
    min_c1 = np.asarray(min_c1, dtype=float)

    model = pulp.LpProblem("Petroleum_Blending", pulp.LpMaximize)

    # x[i, j] = barrels of component i used in grade j per day
    x = model.add_variable_dict("x", (components, grades), lowBound=0, cat="Continuous")

    # profit per barrel = selling price of the grade - cost of the component
    model += pulp.lpSum((price[j] - cost[i]) * x[c, g] for i, c in enumerate(components) for j, g in enumerate(grades))

    # constraints
    for i, c in enumerate(components):
        model += pulp.lpSum(x[c, g] for g in grades) <= availability[i], f"Availability_{c}"

    total = {g: pulp.lpSum(x[c, g] for c in components) for g in grades}
    for j, g in enumerate(grades):
        model += total[g] >= min_production, f"Min_production_{g}"
        model += x["C1", g] >= min_c1[j] * total[g], f"Min_C1_{g}"

    model += x["C2", "Super"] <= max_c2_super * total["Super"], "Max_C2_Super"
    model += x["C3", "Premium"] <= max_c3_premium * total["Premium"], "Max_C3_Premium"
    model += x["C2", "Extra"] >= min_c2_extra * total["Extra"], "Min_C2_Extra"

    result = model.solve(pulp.HiGHS(msg=False))
    return model, x, result

In [4]:
model, x, result = solve_petroleum(components, grades, cost, availability, price, min_production,
                                   min_c1, max_c2_super, max_c3_premium, min_c2_extra)
if result.status_str == "Optimal":
    blend = pd.DataFrame(
        [[x[c, g].value() for g in grades] for c in components],
        index=components, columns=grades,
    )
    print("Optimal blend (barrels/day):")
    print(blend)
    print("\nBarrels of each grade:")
    print(blend.sum())
    print("\nShare of each component:")
    print((blend / blend.sum()).round(3))
    print("\nTotal Profit:", round(result.objective, 2), "euro/day")

Optimal blend (barrels/day):
     Super  Premium   Extra
C1  1500.0   1200.0  1800.0
C2   900.0   1050.0   750.0
C3   600.0    750.0   450.0

Barrels of each grade:
Super      3000.0
Premium    3000.0
Extra      3000.0
dtype: float64

Share of each component:
    Super  Premium  Extra
C1    0.5     0.40   0.60
C2    0.3     0.35   0.25
C3    0.2     0.25   0.15

Total Profit: 76800.0 euro/day


In [5]:
extra_profit = sum((price[2] - cost[i]) * x[c, "Extra"].value() for i, c in enumerate(components))
print("Profit from the Extra grade:", round(extra_profit, 2), "euro/day")

Profit from the Extra grade: 18600.0 euro/day


In [8]:
model.writeMPS("petroleum.mps", with_objsense=True)

h = highspy.Highs()
h.setOptionValue("output_flag", False)
h.readModel("petroleum.mps")
h.run()

lp = h.getLp()
n = lp.num_col_
status, ranging = h.getRanging()

# range of each objective coefficient over which the current basis stays optimal
sensitivity = pd.DataFrame({
    "value": list(h.getSolution().col_value),
    "profit/barrel": list(lp.col_cost_),
    "min profit": list(ranging.col_cost_dn.value_)[:n],
    "max profit": list(ranging.col_cost_up.value_)[:n],
}, index=[h.getColName(i)[1] for i in range(n)])
sensitivity

,value,profit/barrel,min profit,max profit
x_C1_Super,1500.0,11.0,8.0,inf
x_C1_Premium,1200.0,8.0,-inf,11.000000
x_C1_Extra,1800.0,6.0,-inf,17.333333
x_C2_Super,900.0,13.0,13.0,inf
x_C2_Premium,1050.0,10.0,-8.0,10.000000
x_C2_Extra,750.0,8.0,8.0,8.000000
x_C3_Super,600.0,9.0,1.5,9.000000
x_C3_Premium,750.0,6.0,6.0,10.800000
x_C3_Extra,450.0,4.0,4.0,4.000000


In [9]:
# Shadow prices and right-hand-side ranges of the availability constraints (uses h from Cell 8)
sol = h.getSolution()
rows = [h.getRowName(i)[1] for i in range(lp.num_row_)]

availability_analysis = pd.DataFrame({
    "available": list(lp.row_upper_),
    "used": list(sol.row_value),
    "surplus": [u - a for u, a in zip(lp.row_upper_, sol.row_value)],
    "shadow price (€/barrel)": list(sol.row_dual),
    "min availability": list(ranging.row_bound_dn.value_)[:lp.num_row_],
    "max availability": list(ranging.row_bound_up.value_)[:lp.num_row_],
}, index=rows).loc[[f"Availability_{c}" for c in components]]
availability_analysis

,available,used,surplus,shadow price (€/barrel),min availability,max availability
Availability_C1,4500.0,4500.0,0.0,20.0,4500.0,5250.0
Availability_C2,2700.0,2700.0,0.0,4.0,2250.0,3150.0
Availability_C3,3500.0,1800.0,1700.0,0.0,1800.0,2250.0


In [10]:
# Question: component 3 costs 1€ more per barrel
cost_c3 = [12, 10, 15]
model_c3, x_c3, result_c3 = solve_petroleum(components, grades, cost_c3, availability, price, min_production,
                                            min_c1, max_c2_super, max_c3_premium, min_c2_extra)
blend_c3 = pd.DataFrame([[x_c3[c, g].value() for g in grades] for c in components], index=components, columns=grades)
print("New blend (barrels/day):")
print(blend_c3)
print("\nSame composition as before:", blend_c3.equals(blend))
print("Total Profit:", round(result_c3.objective, 2), "€/day (was", round(result.objective, 2), ")")

New blend (barrels/day):
     Super  Premium   Extra
C1  1500.0   1200.0  1800.0
C2   900.0   1050.0   750.0
C3   600.0    750.0   450.0

Same composition as before: True
Total Profit: 75000.0 €/day (was 76800.0 )


In [11]:
# Question: availability of component 2 reduced
for c2_available in [400, 1350, 2300]:
    availability_c2 = [4500, c2_available, 3500]
    model_c2, x_c2, result_c2 = solve_petroleum(components, grades, cost, availability_c2, price, min_production,
                                                min_c1, max_c2_super, max_c3_premium, min_c2_extra)
    print(f"Component 2 available: {c2_available} -> {result_c2.status_str}", end="")
    if result_c2.status_str == "Optimal":
        print(f", profit {result_c2.objective:.0f} €/day")
        print(pd.DataFrame([[x_c2[c, g].value() for g in grades] for c in components], index=components, columns=grades))
    else:
        print()

Component 2 available: 400 -> Infeasible
Component 2 available: 1350 -> Optimal, profit 71400 €/day
     Super  Premium   Extra
C1  1500.0   1200.0  1800.0
C2    -0.0   1050.0   300.0
C3  1500.0    750.0   900.0
Component 2 available: 2300 -> Optimal, profit 75200 €/day
     Super  Premium   Extra
C1  1500.0   1200.0  1800.0
C2   900.0   1050.0   350.0
C3   600.0    750.0   850.0


### Answers to the analysis questions

The solver's optimal solution produces exactly 3000 barrels of each grade, for a total profit of **76,800 €/day**:

| Barrels/day | Super | Premium | Extra |
|---|---|---|---|
| Component 1 | 1500 | 1200 | 1800 |
| Component 2 | 900 | 1050 | 750 |
| Component 3 | 600 | 750 | 450 |

*Note:* this LP has alternative optimal solutions with the same total profit of 76,800 €. Component 1 is used up entirely by the minimum shares, which fixes the totals per grade, but components 2 and 3 can be split between the grades in different ways. Answers 1 and 2 therefore refer to the specific solution returned by the solver (HiGHS); another solver may stop at a different optimal vertex.

---

**1. What is the total profit obtained by the company with the Extra grade with the solution provided by the solver?**

$$(18 - 12) \cdot 1800 + (18 - 10) \cdot 750 + (18 - 14) \cdot 450 = 10{,}800 + 6{,}000 + 1{,}800 = \mathbf{18{,}600\ €/day}$$

Across all optimal solutions, the Extra-grade profit ranges from 16,800 to 20,400 €/day.

---

**2. What is the maximum value that the profit per barrel of component 2, in the Extra grade, can take without changing the base of the optimal solution?**

The current profit per barrel is 18 − 10 = 8 €, and the sensitivity range of this coefficient is [8, 8]. The **maximum value is 8 €**, so the allowable increase is 0.

The reason is the alternative optima: the amount of component 2 in Extra can vary between 300 and 1200 barrels with the same total profit, and the solver's value (750) lies strictly inside that range. If component 2 in Extra earned even slightly more than 8 €, the solutions with 1200 barrels would become strictly better, so the basis changes immediately.

---

**3. If the cost of the barrel of component 3 is increased by 1€, will the optimal composition change?**

**No.** Re-solving with a cost of 15 € gives exactly the same composition; only the profit drops by 1 € × 1800 barrels = 1800 €, to **75,000 €/day**.

- Component 3 is not scarce: its shadow price is 0.
- The amount of component 3 used is fixed by the other constraints. Component 1 is fully used by the minimum shares (0.5·3000 + 0.4·3000 + 0.6·3000 = 4500 barrels), which pins every grade at 3000 barrels. That leaves 4500 barrels for components 2 and 3. All 2700 barrels of component 2 are used, since it is cheaper, so component 3 always fills the remaining 1800 barrels.
- A higher cost therefore lowers the profit of every optimal solution by the same 1800 €, and the choice of composition does not change.

The single-coefficient ranges suggest a basis change, because the coefficients of component 3 in Premium and Extra fall outside their ranges. That is an artefact of the alternative optima; the re-solved model confirms the composition stays the same.

---

**4. Do we use every barrel available of every component in the optimal solution? Or do we have any surplus availability for any component?**

| Component | Available | Used | Surplus | Shadow price |
|---|---|---|---|---|
| 1 | 4500 | 4500 | 0 | 20 €/barrel |
| 2 | 2700 | 2700 | 0 | 4 €/barrel |
| 3 | 3500 | 1800 | **1700** | 0 €/barrel |

Components 1 and 2 are binding constraints: they are fully used, and a positive shadow price means more of them would increase profit. Component 3 has a **surplus of 1700 barrels/day**; its constraint is not binding, so its shadow price is 0 and extra barrels would add nothing.

---

**5. The provider of Component 2 may have a problem with the delivery and the availability may be reduced to 400 barrels. Will this shortage affect the optimal composition?**

**Yes, the problem becomes infeasible**: no production plan meets all the constraints.

- Extra needs at least 10 % component 2: 0.1 · 3000 = **300 barrels**.
- Premium allows at most 25 % component 3, so components 1 and 2 must make up at least 75 % (2250 barrels). Component 1 is already fully used by the minimum shares, so Premium has only 1200 barrels of it and needs at least **1050 barrels** of component 2.
- The minimum feasible availability of component 2 is therefore 300 + 1050 = **1350 barrels**. With 400 barrels the company cannot produce 3000 barrels of each grade.

If the statement means a reduction *by* 400 barrels (to 2300), the problem stays feasible. 2300 is within the range [2250, 3150] where the basis does not change, so the same constraints stay binding but the values change. Profit drops by 400 × 4 € = 1600 € to **75,200 €/day**, and the Extra grade replaces 400 barrels of component 2 with component 3 (350 and 850 barrels instead of 750 and 450).

---

**6. Which component is a better candidate to increase the availability? How would you use the model to negotiate new terms with the provider, assuming you do not want any change in the base of the optimal solution?**

**Component 1** is the best candidate, because it has the highest shadow price:

| Component | Shadow price | Availability range (basis unchanged) | Maximum price worth paying |
|---|---|---|---|
| 1 | 20 €/barrel | 4500 – 5250 (+750) | 12 + 20 = **32 €/barrel** |
| 2 | 4 €/barrel | 2250 – 3150 (+450) | 10 + 4 = 14 €/barrel |
| 3 | 0 €/barrel | surplus of 1700 | no value in more |

How to use the model in the negotiation:

- **Volume:** ask the supplier of component 1 for up to **750 extra barrels/day**. Beyond 5250 barrels the basis changes, and the shadow price is no longer guaranteed.
- **Price:** each extra barrel is worth 20 € of profit on top of its current cost, so the company breaks even at 32 €/barrel. Any price below that is a gain, up to 750 × 20 = **15,000 €/day** extra profit at the current price of 12 €.
- **Component 2:** a second option, worth at most 4 € extra per barrel, for up to 450 extra barrels.
- **Component 3:** 1700 barrels a day go unused, so Grace could instead negotiate a smaller contract or a lower price with this supplier.